# Optimisation multi-objectif (NSGA-II) — Équilibre thermique d'habitation

Reproduction **simplifiée** du notebook d'optimisation multi-objectif mentionné dans
[`BuildingTherm.mo`](BuildingTherm.mo) (sections citées en commentaire : météo, décalage
climatique...). Ce notebook :

- utilise le **même solveur** que l'app Streamlit en secours quand OpenModelica n'est pas
  disponible : [`BuildingTherm.py`](BuildingTherm.py) (réimplémentation pure Python du modèle,
  intégrée avec `scipy.integrate.solve_ivp`, méthode `BDF`) ;
- optimise les **mêmes 5 variables de conception** que les curseurs mis en avant dans l'app
  (Chauffage, Climatisation, Photovolt., Isolation ext., Isolation int.) ;
- calcule les **mêmes indicateurs de sortie** que l'app (température min/max, conso nette,
  autoconsommation, coût €) ;
- lance **NSGA-II** ([`pymoo`](https://pymoo.org/)) pour explorer le compromis
  coût ↔ confort thermique ;
- **regroupe** (k-means) le front de Pareto obtenu en **N scénarios représentatifs**
  (5 par défaut) et affiche, pour chacun, le même graphique temporel que l'app.

Voir le [dépôt GitHub](https://github.com/yannrichet/OptimHome) et le
[README](https://github.com/yannrichet/OptimHome#readme) pour le contexte complet
(modèle physique, app Streamlit, solveur de secours).


## 0. Installation (Google Colab) et récupération du modèle

Sur Colab, cette cellule installe les dépendances qui manquent et télécharge
`BuildingTherm.py` depuis GitHub (le notebook est autonome : il n'a pas besoin du
reste du dépôt cloné). En local, si `BuildingTherm.py` est déjà à côté de ce
notebook, l'installation est simplement ignorée.


In [ ]:
import sys, subprocess, importlib.util, urllib.request

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "pymoo", "scikit-learn", "plotly", "pandas", "numpy", "scipy", "requests"],
        check=True,
    )

if importlib.util.find_spec("BuildingTherm") is None:
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/yannrichet/OptimHome/main/BuildingTherm.py",
        "BuildingTherm.py",
    )

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import requests
from datetime import date, timedelta

import BuildingTherm as bt

print("BuildingTherm.py chargé.")


## 1. Météo réelle (Open-Meteo)

Même source et même API que l'app (`archive-api.open-meteo.com`, réanalyse ERA5,
sans clé). Position et période par défaut ci-dessous — à changer librement.


In [ ]:
LAT, LON = 48.8566, 2.3522          # Paris par défaut ; change librement
END_DATE = date.today()
START_DATE = END_DATE - timedelta(days=364)
WARMUP_DAYS = 14                     # mise en régime, comme dans l'app


def fetch_weather(lat, lon, start_date, end_date, warmup_days=WARMUP_DAYS):
    """(times, Tout_K, Gh) horaires, même format que BuildingTherm.load_weather_table."""
    fetch_start = start_date - timedelta(days=warmup_days)
    r = requests.get(
        "https://archive-api.open-meteo.com/v1/archive",
        params={"latitude": lat, "longitude": lon,
                "start_date": fetch_start.isoformat(), "end_date": end_date.isoformat(),
                "hourly": "temperature_2m,shortwave_radiation", "timezone": "UTC"},
        timeout=60,
    )
    r.raise_for_status()
    h = r.json()["hourly"]
    Tout_K = np.array(h["temperature_2m"]) + 273.15
    Gh = np.maximum(np.array(h["shortwave_radiation"]), 0.0)
    times = np.arange(len(Tout_K)) * 3600.0
    return times, Tout_K, Gh


weather = fetch_weather(LAT, LON, START_DATE, END_DATE)
stop_time = (len(weather[0]) - 1) * 3600.0
print(f"{len(weather[0])} points horaires ({stop_time/86400:.1f} j, dont {WARMUP_DAYS} j de mise en régime)")


## 2. Variables de conception, objectifs, fonction de simulation

**Variables de conception** (5, mêmes bornes que les sliders de l'app) :
`Pheat` (Chauffage), `Pcool` (Climatisation), `Ppv_kWc` (Photovolt.),
`e_ite_cm` (Isolation ext.), `e_iti_cm` (Isolation int.).

**Objectifs** (à minimiser tous les deux, mêmes indicateurs que l'app) :
- **Coût net** [€] : `Egrid_cool·prix_elec − Eexport·prix_rachat_pv`, sur la période choisie.
- **Inconfort** [°C cumulés] : dépassement de la bande de confort
  `max(0, Tconfort_min − Tmin) + max(0, Tmax − Tconfort_max)`.

Tous les autres paramètres du modèle restent aux valeurs par défaut de l'app
(matériau parpaing, géométrie 40 m²/7,5 m, ventilation, etc.).


In [ ]:
FIXED_PARAMS = dict(
    ach_day=1.5, ach_night=2.0,
    lam_iso=0.036, ach=0.6, Qint=400.0, dTout=1.0, fsol=0.5, seer=3.5,
    Sfloor=40.0, Htot=7.5, Awin=20.0, UAother_ref=58.0, Sfloor_ref=40.0,
    e_blk=0.20, lam_blk=0.95, rhoc_blk=1300 * 1000.0, rhoc_iso=30 * 1030.0,
    hi=7.7, he=25.0,
    Tset_h=292.15, Tset_c=299.15, Kp=4000.0, Kc=4000.0,
    fanWhm3=0.15, PR_pv=0.90,
)
T_CONFORT_MIN, T_CONFORT_MAX = 19.0, 26.0    # °C, mêmes valeurs par défaut que l'app
PRIX_ELEC, PRIX_RACHAT_PV = 0.2516, 0.04     # €/kWh, mêmes valeurs par défaut que l'app
WARMUP_HOURS = WARMUP_DAYS * 24

VAR_NAMES = ["Pheat", "Pcool", "Ppv_kWc", "e_ite_cm", "e_iti_cm"]
VAR_LABELS = ["Chauffage [W]", "Climatisation [W]", "Photovolt. [kWc]",
              "Isolation ext. [cm]", "Isolation int. [cm]"]
XL = np.array([1000.0, 0.0, 0.0, 0.0, 0.0])       # mêmes bornes que les sliders de l'app
XU = np.array([12000.0, 6000.0, 9.0, 30.0, 20.0])


def simulate_scenario(x):
    """x = [Pheat, Pcool, Ppv_kWc, e_ite_cm, e_iti_cm] -> DataFrame (colonnes = app)."""
    Pheat, Pcool, Ppv_kWc, e_ite_cm, e_iti_cm = x
    params = dict(FIXED_PARAMS, Pheat=Pheat, Pcool=Pcool, Ppv_kWc=Ppv_kWc,
                  e_ite=e_ite_cm / 100.0, e_iti=e_iti_cm / 100.0)
    rows = bt.simulate(params, weather, stop_time)
    return pd.DataFrame(rows)


def objectives(x):
    """(cout_net_eur, inconfort_degres) sur la période, après mise en régime."""
    sim = simulate_scenario(x)
    sim_p = sim.iloc[WARMUP_HOURS:] if len(sim) > WARMUP_HOURS else sim
    Tmin = sim_p["Tair"].min() - 273.15
    Tmax = sim_p["Tair"].max() - 273.15
    egrid_cool = sim["Egrid_cool"].iloc[-1]
    eexport = sim["Eexport"].iloc[-1]
    cout = egrid_cool * PRIX_ELEC - eexport * PRIX_RACHAT_PV
    inconfort = max(0.0, T_CONFORT_MIN - Tmin) + max(0.0, Tmax - T_CONFORT_MAX)
    return cout, inconfort


# sanity check sur un scenario "par defaut" (memes valeurs que l'app au chargement)
print("objectifs (defaut app) :", objectives([8000, 2000, 3.0, 16.0, 0.0]))


## 3. Optimisation multi-objectif (NSGA-II, `pymoo`)

`POP_SIZE` × `N_GEN` simulations annuelles complètes seront exécutées (chacune
~4-5 s en pur Python) : avec les valeurs par défaut ci-dessous, compter
quelques minutes. Augmenter ces deux valeurs donne un front de Pareto plus
fin, au prix du temps de calcul (linéaire).


In [ ]:
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.optimize import minimize


class BuildingProblem(Problem):
    def __init__(self):
        super().__init__(n_var=5, n_obj=2, xl=XL, xu=XU)

    def _evaluate(self, X, out, *args, **kwargs):
        out["F"] = np.array([objectives(x) for x in X])


POP_SIZE = 16   # augmenter (ex. 40) pour un front plus fin
N_GEN = 6        # augmenter (ex. 15) pour une meilleure convergence

algorithm = NSGA2(
    pop_size=POP_SIZE,
    sampling=FloatRandomSampling(),
    crossover=SBX(prob=0.9, eta=15),
    mutation=PM(eta=20),
)

res = minimize(BuildingProblem(), algorithm, ("n_gen", N_GEN), seed=1, verbose=True)
print(f"{len(res.F)} solutions non dominées sur le front de Pareto final")


## 4. Regroupement du front de Pareto en N scénarios représentatifs

K-means (dans l'espace des objectifs, standardisé) découpe le front en `N_SCENARIOS`
groupes ; pour chacun, on retient la solution la plus proche du centroïde comme
scénario représentatif.


In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

N_SCENARIOS = 5   # nombre de scenarios a extraire du front de Pareto

F, X = res.F, res.X   # F : (n_sol, 2) [cout_net_eur, inconfort] ; X : (n_sol, 5) variables

scaler = StandardScaler()
F_scaled = scaler.fit_transform(F)

k = min(N_SCENARIOS, len(F))
kmeans = KMeans(n_clusters=k, n_init=10, random_state=0).fit(F_scaled)

selected_idx = []
for c in range(k):
    members = np.where(kmeans.labels_ == c)[0]
    center = kmeans.cluster_centers_[c]
    dists = np.linalg.norm(F_scaled[members] - center, axis=1)
    selected_idx.append(members[np.argmin(dists)])
selected_idx = sorted(selected_idx, key=lambda i: F[i, 0])  # tri par cout croissant

scenarios_df = pd.DataFrame(X[selected_idx], columns=VAR_NAMES)
scenarios_df["Cout_net_eur"] = F[selected_idx, 0]
scenarios_df["Inconfort_degres"] = F[selected_idx, 1]
scenarios_df.index = [f"Scénario {i + 1}" for i in range(len(selected_idx))]
scenarios_df.round(1)


## 5. Front de Pareto : coût vs inconfort

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=F[:, 1], y=F[:, 0], mode="markers", name="Front de Pareto",
    marker=dict(color="rgba(31,119,180,0.5)", size=8),
))
fig.add_trace(go.Scatter(
    x=F[selected_idx, 1], y=F[selected_idx, 0], mode="markers+text", name="Scénarios retenus",
    marker=dict(color="rgba(214,39,40,0.9)", size=14, symbol="diamond"),
    text=[f"S{i + 1}" for i in range(len(selected_idx))], textposition="top center",
))
fig.update_layout(
    xaxis_title="Inconfort cumulé [°C]", yaxis_title="Coût net [€]",
    title="Front de Pareto coût / inconfort — scénarios représentatifs en évidence",
    height=450, margin=dict(l=60, r=30, t=50, b=50),
)
fig.show()


## 6. Détail temporel des scénarios sélectionnés

Même visualisation que l'app Streamlit (bande de confort, températures
intérieure/extérieure min-max journalières, puissances importées/autoconsommées).


In [ ]:
def plot_scenario(x, title):
    sim = simulate_scenario(x)
    sim_p = sim.iloc[WARMUP_HOURS:].reset_index(drop=True)
    jours = ((sim_p["time"] - sim_p["time"].iloc[0]) / 86400).astype(int)

    daily = pd.DataFrame({
        "jour": jours,
        "Tint_min": sim_p["Tair"] - 273.15, "Tint_max": sim_p["Tair"] - 273.15,
        "Text_min": sim_p["Tout"] - 273.15, "Text_max": sim_p["Tout"] - 273.15,
    }).groupby("jour").agg({"Tint_min": "min", "Tint_max": "max", "Text_min": "min", "Text_max": "max"})
    kW_grid = (sim_p["Pgrid_cool"] / 1000).groupby(jours).mean()
    kW_pv_self = (sim_p["Pself_cool"] / 1000).groupby(jours).mean()

    fig = go.Figure()
    fig.add_hrect(y0=T_CONFORT_MIN, y1=T_CONFORT_MAX, fillcolor="rgba(46,160,67,0.12)", line_width=0,
                  annotation_text=f"confort {T_CONFORT_MIN:.0f}-{T_CONFORT_MAX:.0f} °C", annotation_position="top left")
    fig.add_trace(go.Scatter(x=daily.index, y=daily["Text_max"], mode="lines",
                              line=dict(width=1.2, color="rgba(120,120,120,0.9)"), name="T extérieure max/j"))
    fig.add_trace(go.Scatter(x=daily.index, y=daily["Text_min"], mode="lines",
                              line=dict(width=1.2, color="rgba(120,120,120,0.9)", dash="dot"),
                              fill="tonexty", fillcolor="rgba(120,120,120,0.25)", name="T extérieure min/j"))
    fig.add_trace(go.Scatter(x=daily.index, y=daily["Tint_max"], mode="lines",
                              line=dict(width=1.2, color="rgba(31,119,180,0.9)"), name="T intérieure max/j"))
    fig.add_trace(go.Scatter(x=daily.index, y=daily["Tint_min"], mode="lines",
                              line=dict(width=1.2, color="rgba(31,119,180,0.9)", dash="dot"),
                              fill="tonexty", fillcolor="rgba(31,119,180,0.3)", name="T intérieure min/j"))
    fig.add_trace(go.Scatter(x=kW_grid.index, y=kW_grid.values, mode="lines", name="Import réseau [kW]",
                              stackgroup="power", yaxis="y2", line=dict(color="rgba(214,39,40,0.9)", width=0.5),
                              fillcolor="rgba(214,39,40,0.35)"))
    fig.add_trace(go.Scatter(x=kW_pv_self.index, y=kW_pv_self.values, mode="lines", name="Autoconso PV [kW]",
                              stackgroup="power", yaxis="y2", line=dict(color="rgba(255,127,14,0.9)", width=0.5),
                              fillcolor="rgba(255,127,14,0.35)"))
    fig.update_layout(
        title=title, xaxis_title="Jour", yaxis=dict(title="Température [°C]"),
        yaxis2=dict(title="Puissance moyenne/j [kW]", overlaying="y", side="right", rangemode="tozero"),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        height=380, margin=dict(l=60, r=60, t=60, b=40), hovermode="x unified",
    )
    return fig


for i, idx in enumerate(selected_idx):
    x = X[idx]
    params_txt = ", ".join(f"{n}={v:.1f}" for n, v in zip(VAR_NAMES, x))
    title = (f"Scénario {i + 1}/{len(selected_idx)} — {params_txt}<br>"
             f"Coût net {F[idx, 0]:.0f} € · Inconfort {F[idx, 1]:.1f} °C cumulés")
    plot_scenario(x, title).show()
